In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.neighbors import KernelDensity

DATA_DIR = Path("../results")
RESULTS_DIR = Path("../figures")
CORR_DIR = DATA_DIR / "correlation"

In [2]:
def plot_kda_heatmap(data, x_col, y_col, title, xlabel, ylabel, filename, *, x_range=0, y_range=1):
    kde = KernelDensity(kernel='gaussian', bandwidth=0.003)
    kde.fit(data[[x_col, y_col]].values)

    plt.figure(figsize=(6, 6), dpi=300)
    N = 150
    x = np.linspace(x_range[0], x_range[1], N)
    y = np.linspace(y_range[0], y_range[1], N)
    X, Y = np.meshgrid(x, y)

    Z = np.exp(kde.score_samples(np.vstack([X.ravel(), Y.ravel()]).T)).reshape(X.shape)
    plt.contourf(X, Y * 100, Z, levels=20, cmap='Reds')

    dx = x[1] - x[0]
    dy = y[1] - y[0]
    cell_area = dx * dy

    z_flat = Z.ravel()
    z_sorted = np.sort(z_flat)[::-1]
    cumulative_mass = np.cumsum(z_sorted * cell_area)
    cumulative_mass /= cumulative_mass[-1]
    ci_levels = [0.50, 0.80, 0.90, 0.95]
    density_thresholds = []

    for ci in ci_levels:
        idx = np.searchsorted(cumulative_mass, ci)
        threshold = z_sorted[min(idx, len(z_sorted) - 1)]
        density_thresholds.append(threshold)

    alphas = [1.0, 0.8, 0.6, 0.4]

    for level, alpha, label in zip(density_thresholds, alphas, ci_levels):
        plt.contour(
            X, Y * 100, Z,
            levels=[level],
            colors='blue',
            linewidths=1,
            alpha=alpha
        )


    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid()  
    plt.tight_layout()
    plt.savefig(RESULTS_DIR.joinpath(filename))
    plt.close()

In [3]:
partial_correlation = pd.read_csv(CORR_DIR.joinpath("correlation_z_mratio.csv"))
partial_correlation['Difference'] = partial_correlation['Pearson r'] - partial_correlation['Semipartial r']
partial_correlation['Reduction'] = -(partial_correlation['Difference'] / partial_correlation['Pearson r'])

plot_kda_heatmap(
    data=partial_correlation,
    x_col='Pearson r',
    y_col='Reduction',
    title='Reduction of correlation - Moment Ratio',
    xlabel='Reduction of correlation [%]',
    ylabel='Correlation',
    filename="figure_4a.pdf",
    x_range=(-0.81, -0.74),
    y_range=(-1, 0),
)

partial_correlation = pd.read_csv(CORR_DIR.joinpath("correlation_z_rout.csv"))
partial_correlation['Difference'] = partial_correlation['Pearson r'] - partial_correlation['Semipartial r']
partial_correlation['Reduction'] = -(partial_correlation['Difference'] / partial_correlation['Pearson r'])

plot_kda_heatmap(
    data=partial_correlation,
    x_col='Pearson r',
    y_col='Reduction',
    title='Reduction of correlation - Rout',
    xlabel='Reduction of correlation [%]',
    ylabel='Correlation',
    filename="figure_4b.pdf",
    x_range=(-0.81, -0.74),
    y_range=(-1, 0),
)